# Notebook 02 — Traps in Li₂O

**Learning path:** `01_resistance_network` → **`02_traps`** → `03_multi_material` → `04_temperature_gradient` → `05_pebble_3D`

---

## What you'll learn

- Why trapping matters for tritium retention in breeder blankets
- How to switch from steady-state to transient and what changes
- How to add a single trap to the Li₂O model using `F.Reaction` + `F.ImplicitSpecies`
- How to extract and plot inventory and surface flux over time
- How to use the convenience class `F.Trap` for simpler cases
- How to add a second trap (e.g. radiation-induced defects)

---

## Background: Why do traps matter?

In a real Li₂O breeder pebble, tritium produced by the ${}^6\text{Li}(n,\alpha)\text{T}$ reaction doesn't only diffuse freely through the lattice. A fraction of the tritium atoms get temporarily (or permanently) caught at **defect sites** — grain boundaries, vacancies, dislocations, or impurity atoms. These are called **traps**.

Trapping has two major consequences:

1. **Increased retention** — trapped tritium cannot be swept out by the purge gas. At steady state, the *inventory* (total T in the solid) is much higher than the mobile inventory alone.
2. **Delayed breakthrough** — a pebble with traps takes much longer to reach the steady-state permeation flux than a trap-free pebble. The ratio of timescales is the **retardation factor** $R = 1 + K_{eq} \cdot n_t$.

In the resistance-network picture, trapping makes the **effective diffusivity** much smaller:
$$D_{\text{eff}} = \frac{D}{1 + K_{eq} \cdot n_t} = \frac{D}{R}$$

The trapping/detrapping reaction is:
$$\text{T}_{\text{mobile}} + [\,] \underset{p(T)}{\overset{k(T)}{\rightleftharpoons}} [\text{T}]$$

with Arrhenius rates:
$$k(T) = k_0 \exp\!\left(-\frac{E_k}{k_B T}\right), \qquad p(T) = p_0 \exp\!\left(-\frac{E_p}{k_B T}\right)$$

The equilibrium constant is $K_{eq}(T) = k(T)/p(T)$. High $E_p$ means traps are hard to escape → large $K_{eq}$ → strong retardation. Higher temperature **reduces** $K_{eq}$ (traps release more easily).

The coupled PDEs FESTIM solves are:
$$\frac{\partial c_m}{\partial t} = \nabla\cdot(D \nabla c_m) - k\, c_m\, n_{\text{empty}} + p\, c_t$$
$$\frac{\partial c_t}{\partial t} = k\, c_m\, n_{\text{empty}} - p\, c_t$$

where $n_{\text{empty}} = n_t - c_t$ (implicit species in FESTIM).

---
## 1. Imports and shared geometry

Everything here is identical to your working steady-state model.

In [1]:
import festim as F
import numpy as np
import matplotlib.pyplot as plt

# ── Geometry ──────────────────────────────────────────────────────────────────
L = 1e-3                          # slab thickness [m]
vertices = np.linspace(0, L, 100) # uniform 1D mesh

# ── Material: Li₂O (Tanifuji 1987) ───────────────────────────────────────────
li2o = F.Material(D_0=1.16e-5, E_D=1.047)   # D in m²/s, E_D in eV

# ── Temperature ───────────────────────────────────────────────────────────────
T_K = 773.0   # 500 °C — mid-range of Tanifuji validity window

# ── Boundary conditions (same Sievert + zero-flux as in Notebook 01) ──────────
P_up = 1000.0   # upstream tritium partial pressure [Pa]
S_0  = 1.0e20   # Sievert pre-exponential [mol/m³/Pa^0.5]  ← placeholder value
E_S  = 0.5      # Sievert activation energy [eV]

# ── Useful constants ──────────────────────────────────────────────────────────
kB = 8.617e-5   # eV/K

# Quick sanity-check of the diffusion time scale
D_val = li2o.D_0 * np.exp(-li2o.E_D / (kB * T_K))
tau_diff = L**2 / D_val
print(f"D(773 K)  = {D_val:.3e} m²/s")
print(f"τ_diff    = L²/D = {tau_diff:.3e} s  ({tau_diff/86400:.1f} days)")
print()
print("Note: Li₂O diffusion is very slow at 500 °C.")
print("We will run to final_time >> τ_diff to capture the full transient.")

D(773 K)  = 1.730e-12 m²/s
τ_diff    = L²/D = 5.781e+05 s  (6.7 days)

Note: Li₂O diffusion is very slow at 500 °C.
We will run to final_time >> τ_diff to capture the full transient.


---
## 2. Model A — transient, NO traps (baseline)

Before adding traps we need to understand what the **trap-free transient** looks like.  
The only change from Notebook 01 is:

- `transient=True` in `F.Settings`  
- `final_time` and a `Stepsize` object  
- Exports: `SurfaceFlux` (breakthrough curve) and `TotalVolume` (inventory)

In [ ]:
# ── Build Model A ─────────────────────────────────────────────────────────────
model_A = F.HydrogenTransportProblem()
model_A.mesh = F.Mesh1D(vertices)

vol_A       = F.VolumeSubdomain1D(id=1, borders=[0, L], material=li2o)
surf_left_A = F.SurfaceSubdomain1D(id=1, x=0)
surf_right_A= F.SurfaceSubdomain1D(id=2, x=L)
model_A.subdomains = [vol_A, surf_left_A, surf_right_A]

tritium_A = F.Species('T')
model_A.species = [tritium_A]

model_A.temperature = T_K

model_A.boundary_conditions = [
    F.SievertsBC(
        subdomain=surf_left_A, S_0=S_0, E_S=E_S,
        pressure=P_up, species=tritium_A,
    ),
    F.DirichletBC(
        subdomain=surf_right_A, value=0.0, species=tritium_A,
    ),
]

# ── Exports ───────────────────────────────────────────────────────────────────
# Right-surface flux (breakthrough curve) — negative = leaving the slab
flux_A     = F.SurfaceFlux(surface=surf_right_A, field=tritium_A)
# Total mobile inventory [mol] integrated over the slab volume
inv_A      = F.TotalVolume(field=tritium_A, volume=vol_A)
# Concentration profiles snapped at four times
snap_times = [tau_diff * f for f in [0.1, 0.3, 0.6, 1.0]]
prof_A     = F.Profile1DExport(field=tritium_A, times=snap_times)

model_A.exports = [flux_A, inv_A, prof_A]

# ── Settings: transient ───────────────────────────────────────────────────────
# Run for ~2 diffusion times so we clearly see the approach to steady state
final_time_A = 2 * tau_diff
model_A.settings = F.Settings(
    atol=1e10, rtol=1e-10,
    transient=True,
    final_time=final_time_A,
)
# Adaptive stepsize: start small, grow quickly, cap at τ_diff/20
model_A.settings.stepsize = F.Stepsize(
    initial_value=tau_diff * 1e-3,
    growth_factor=1.3,
    cutback_factor=0.8,
    target_nb_iterations=4,
    max_stepsize=tau_diff / 20,
)

print(f"Running Model A (no traps) to t = {final_time_A:.2e} s ...")
model_A.initialise()
model_A.run()
print("Model A done.")

### Plot Model A results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── Left: concentration profiles ──────────────────────────────────────────────
ax = axes[0]
for i, (t_snap, c_snap) in enumerate(zip(prof_A.t, prof_A.data)):
    frac = t_snap / tau_diff
    ax.plot(prof_A.x * 1e3, c_snap,
            label=f"t = {frac:.1f}·τ")
ax.set(xlabel='x [mm]', ylabel='c_m  [mol/m³]',
       title='Mobile concentration (no traps)')
ax.legend(fontsize=8)

# ── Middle: inventory vs time ─────────────────────────────────────────────────
ax = axes[1]
ax.plot(np.array(inv_A.t) / tau_diff, inv_A.data, color='steelblue')
ax.set(xlabel='t / τ_diff', ylabel='Mobile inventory  [mol/m²]',
       title='Inventory build-up (no traps)')
ax.axhline(inv_A.data[-1], ls='--', color='grey', lw=0.8, label='Steady state')
ax.legend(fontsize=8)

# ── Right: breakthrough flux ──────────────────────────────────────────────────
ax = axes[2]
ax.plot(np.array(flux_A.t) / tau_diff, np.abs(flux_A.data), color='tomato')
ax.set(xlabel='t / τ_diff', ylabel='|Flux|  [mol/m²/s]',
       title='Breakthrough flux (no traps)')

plt.tight_layout()
plt.savefig('A_no_traps.png', dpi=120)
plt.show()
print("Steady-state flux (no traps):", f"{np.abs(flux_A.data[-1]):.4e} mol/m²/s")

---
## 3. Model B — transient, WITH one trap

### Trap parameters and what they mean

We use the **McNabb–Foster** trapping model. Each parameter:

| Parameter | Symbol | Typical value | Meaning |
|-----------|--------|---------------|---------|
| `k_0`     | $k_0$  | ≈ $D_0/(a^2 N_i)$ | Trapping pre-exponential [m³/(mol·s)] |
| `E_k`     | $E_k$  | = $E_D$ (often) | Trapping activation energy [eV] |
| `p_0`     | $p_0$  | ≈ 10¹³ s⁻¹   | Debye frequency (detrapping attempt rate) |
| `E_p`     | $E_p$  | $E_D + E_b$   | Detrapping energy = diffusion barrier + trap binding energy [eV] |
| `n`       | $n_t$  | material-dep. | Total trap density [mol/m³] |

The **retardation factor** at temperature $T$ is:
$$R(T) = 1 + \frac{k(T)}{p(T)}\, n_t = 1 + K_{eq}(T)\, n_t$$

For our demonstration we choose parameters so that $R \approx 10$ at 773 K:  
- $E_p = 1.65$ eV (trap binding energy + diffusion barrier)  
- $n_t = 100$ mol/m³ (≈ 0.15% of Li₂O formula-unit density — plausible for irradiated Li₂O)

We use `F.ImplicitSpecies` to represent the empty trap sites. FESTIM tracks $n_{\text{empty}} = n_t - c_t$ automatically.

In [ ]:
# ── Trap parameters ────────────────────────────────────────────────────────────
k_0_trap = 1e8      # trapping rate pre-exponential [m³/(mol·s)]
E_k_trap = 1.047    # = E_D: no extra barrier above diffusion [eV]
p_0_trap = 1e13     # Debye frequency [s⁻¹]
E_p_trap = 1.653    # detrapping energy [eV]  (trap binding energy ≈ 0.6 eV)
n_traps  = 100.0    # trap density [mol/m³]

# ── Verify retardation factor at 773 K ────────────────────────────────────────
k_773 = k_0_trap * np.exp(-E_k_trap / (kB * T_K))
p_773 = p_0_trap * np.exp(-E_p_trap / (kB * T_K))
K_eq  = k_773 / p_773
R     = 1 + K_eq * n_traps
print(f"k(773 K)  = {k_773:.3e} m³/(mol·s)")
print(f"p(773 K)  = {p_773:.3e} s⁻¹")
print(f"K_eq      = k/p = {K_eq:.3e} m³/mol")
print(f"R = 1 + K_eq·nₜ = {R:.1f}   (effective diffusion {R:.0f}× slower)")
print(f"Expected τ_eff  ≈ R·τ_diff = {R*tau_diff:.2e} s  ({R*tau_diff/86400:.0f} days)")

# Local equilibration time for traps (should be << τ_diff)
# At x=0, c_m ≈ K_S*sqrt(P): traps equilibrate in ~1/(k*c_m + p) seconds
K_S_773    = S_0 * np.exp(-E_S / (kB * T_K))
c_m_left   = K_S_773 * np.sqrt(P_up)
tau_trap   = 1.0 / (k_773 * c_m_left + p_773)
print(f"\nTrap equilibration time ≈ {tau_trap:.2e} s  (should be << τ_diff = {tau_diff:.2e} s)")
print("→ Local equilibrium holds: traps follow c_m instantaneously.")

In [ ]:
# ── Build Model B ─────────────────────────────────────────────────────────────
model_B = F.HydrogenTransportProblem()
model_B.mesh = F.Mesh1D(vertices)

vol_B        = F.VolumeSubdomain1D(id=1, borders=[0, L], material=li2o)
surf_left_B  = F.SurfaceSubdomain1D(id=1, x=0)
surf_right_B = F.SurfaceSubdomain1D(id=2, x=L)
model_B.subdomains = [vol_B, surf_left_B, surf_right_B]

# ── Species ───────────────────────────────────────────────────────────────────
# Mobile tritium — diffuses through the lattice
tritium_B   = F.Species('T')
# Trapped tritium — immobile, created when T is captured by a defect
trapped_T   = F.Species('T_trapped', mobile=False)
# Empty trap sites — their concentration = n_traps - c_trapped (implicit)
empty_traps = F.ImplicitSpecies(n=n_traps, others=[trapped_T])

# IMPORTANT: only put EXPLICIT species in model.species
model_B.species = [tritium_B, trapped_T]

# ── Reaction: T + [ ] ⇌ [T] ──────────────────────────────────────────────────
trap_reaction = F.Reaction(
    reactant=[tritium_B, empty_traps],  # T_mobile + empty site
    product =[trapped_T],               # → trapped T
    k_0=k_0_trap, E_k=E_k_trap,        # forward  (trapping)
    p_0=p_0_trap, E_p=E_p_trap,        # backward (detrapping)
    volume=vol_B,
)
model_B.reactions = [trap_reaction]

model_B.temperature = T_K

model_B.boundary_conditions = [
    F.SievertsBC(
        subdomain=surf_left_B, S_0=S_0, E_S=E_S,
        pressure=P_up, species=tritium_B,
    ),
    F.DirichletBC(
        subdomain=surf_right_B, value=0.0, species=tritium_B,
    ),
]

# ── Exports ───────────────────────────────────────────────────────────────────
flux_B     = F.SurfaceFlux(surface=surf_right_B, field=tritium_B)
inv_B_mob  = F.TotalVolume(field=tritium_B, volume=vol_B)   # mobile inventory
inv_B_trap = F.TotalVolume(field=trapped_T,  volume=vol_B)   # trapped inventory
# Profile snapshots at same FRACTION of τ_eff as Model A used for τ_diff
tau_eff     = R * tau_diff
snap_times_B= [tau_eff * f for f in [0.1, 0.3, 0.6, 1.0]]
prof_B_mob  = F.Profile1DExport(field=tritium_B, times=snap_times_B)
prof_B_trap = F.Profile1DExport(field=trapped_T,  times=snap_times_B)

model_B.exports = [flux_B, inv_B_mob, inv_B_trap, prof_B_mob, prof_B_trap]

# ── Settings ──────────────────────────────────────────────────────────────────
# Run to 2·τ_eff to capture the full trapped transient
final_time_B = 2 * tau_eff
model_B.settings = F.Settings(
    atol=1e10, rtol=1e-10,
    transient=True,
    final_time=final_time_B,
)
model_B.settings.stepsize = F.Stepsize(
    initial_value=tau_diff * 1e-3,
    growth_factor=1.3,
    cutback_factor=0.8,
    target_nb_iterations=4,
    max_stepsize=tau_eff / 20,
)

print(f"Running Model B (with 1 trap, R={R:.0f}) to t = {final_time_B:.2e} s ...")
model_B.initialise()
model_B.run()
print("Model B done.")

### Plot Model B — concentration profiles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = plt.cm.Blues(np.linspace(0.4, 1.0, len(prof_B_mob.t)))

for i, (t_snap, c_mob, c_tr) in enumerate(
        zip(prof_B_mob.t, prof_B_mob.data, prof_B_trap.data)):
    frac = t_snap / tau_eff
    lbl  = f"t = {frac:.1f}·τ_eff"
    axes[0].plot(prof_B_mob.x * 1e3, c_mob,  color=colors[i], label=lbl)
    axes[1].plot(prof_B_mob.x * 1e3, c_tr,   color=colors[i], label=lbl)

axes[0].set(xlabel='x [mm]', ylabel='c_mobile  [mol/m³]', title='Mobile tritium')
axes[1].set(xlabel='x [mm]', ylabel='c_trapped [mol/m³]', title='Trapped tritium')
for ax in axes:
    ax.legend(fontsize=8)

plt.suptitle(f'Model B — 1 trap, R = {R:.0f}, T = {T_K:.0f} K', fontsize=11)
plt.tight_layout()
plt.savefig('B_profiles.png', dpi=120)
plt.show()

# Print ratio of trapped to mobile at final state (should be ≈ K_eq * n_traps = R-1)
c_m_final  = prof_B_mob.data[-1]
c_t_final  = prof_B_trap.data[-1]
midpoint   = len(c_m_final) // 2
ratio      = c_t_final[midpoint] / c_m_final[midpoint] if c_m_final[midpoint] > 0 else 0
print(f"c_trapped / c_mobile at mid-slab (final) ≈ {ratio:.2f}")
print(f"Expected K_eq · n_t = {K_eq * n_traps:.2f}  (should match above)")

---
## 4. Comparison: no traps vs with traps

The two most important engineering quantities are the **inventory** and the **breakthrough flux**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Inventory comparison ──────────────────────────────────────────────────────
ax = axes[0]
t_A = np.array(inv_A.t)
t_B = np.array(inv_B_mob.t)

ax.plot(t_A / tau_diff, inv_A.data,
        color='steelblue', label='Mobile only (no traps)')
ax.plot(t_B / tau_diff,
        np.array(inv_B_mob.data) + np.array(inv_B_trap.data),
        color='tomato',    label=f'Mobile + trapped  (R={R:.0f})')
ax.plot(t_B / tau_diff, inv_B_mob.data,
        color='tomato', ls='--', label='Mobile only (trapped model)', lw=0.8)

ax.set(xlabel='t / τ_diff  (normalised to no-trap timescale)',
       ylabel='Inventory  [mol/m²]',
       title='Total inventory')
ax.legend(fontsize=8)
ax.set_xlim(left=0)

# ── Breakthrough flux comparison ──────────────────────────────────────────────
ax = axes[1]
ax.plot(t_A / tau_diff, np.abs(flux_A.data),
        color='steelblue', label='No traps')
ax.plot(t_B / tau_diff, np.abs(flux_B.data),
        color='tomato',    label=f'With 1 trap  (R={R:.0f})')

ax.set(xlabel='t / τ_diff',
       ylabel='|Breakthrough flux|  [mol/m²/s]',
       title='Breakthrough flux (right surface)')
ax.legend(fontsize=8)
ax.set_xlim(left=0)

plt.suptitle('Effect of trapping on tritium transport', fontsize=11)
plt.tight_layout()
plt.savefig('compare_traps.png', dpi=120)
plt.show()

# ── Key numbers ───────────────────────────────────────────────────────────────
inv_A_ss = inv_A.data[-1]
inv_B_ss = inv_B_mob.data[-1] + inv_B_trap.data[-1]
J_A_ss   = np.abs(flux_A.data[-1])
J_B_ss   = np.abs(flux_B.data[-1])

print("\n── Steady-state comparison ────────────────────────────────────")
print(f"{'Quantity':<35}  {'No traps':>14}  {'With traps':>14}  {'Ratio':>8}")
print("-" * 75)
print(f"{'Permeation flux  [mol/m²/s]':<35}  {J_A_ss:14.4e}  {J_B_ss:14.4e}  {J_B_ss/J_A_ss:8.3f}")
print(f"{'Total inventory  [mol/m²]':<35}  {inv_A_ss:14.4e}  {inv_B_ss:14.4e}  {inv_B_ss/inv_A_ss:8.1f}")
print()
print("Key insight: At steady state, the FLUX is the same with or without traps.")
print("Trapping only increases inventory and slows the approach to steady state.")

---
## 5. Convenience class: `F.Trap`

For the simple case of one mobile species + one trap, FESTIM provides a shorthand.  
This is identical to the `F.Reaction` + `F.ImplicitSpecies` approach above — just fewer lines.

```python
# ── F.Trap shorthand (equivalent to Model B above) ────────────────────────────
tritium_C = F.Species('T')
model_C.species = [tritium_C]   # <- only the MOBILE species here

model_C.traps = [
    F.Trap(
        mobile_species=tritium_C,
        k_0=k_0_trap, E_k=E_k_trap,
        p_0=p_0_trap, E_p=E_p_trap,
        volume=vol_C,
        n=n_traps,       # total trap density [mol/m³]
        name='T_trap1',  # optional label
    )
]
```

**Use `F.Trap` when:** one mobile species, one H per trap, no coupling between traps.  
**Use `F.Reaction` when:** multiple species share a trap, multi-occupancy traps, or you need to access the trapped-species concentration for exports.

---
## 6. Model C — two traps

In a real breeding blanket, Li₂O will have at least two trap populations:

| Trap | Origin | $E_p$ | $n_t$ | Effect |
|------|--------|--------|-------|--------|
| **Trap 1** (intrinsic) | Stoichiometry defects, grain boundaries | lower | lower | Moderate retardation |
| **Trap 2** (irradiation) | Neutron-displacement damage | higher | higher | Strong retardation, hard to release |

Two `F.Reaction` objects with two `F.ImplicitSpecies` — FESTIM assembles the full coupled PDE automatically.

In [ ]:
# ── Build Model C: two traps ───────────────────────────────────────────────────
model_C = F.HydrogenTransportProblem()
model_C.mesh = F.Mesh1D(vertices)

vol_C        = F.VolumeSubdomain1D(id=1, borders=[0, L], material=li2o)
surf_left_C  = F.SurfaceSubdomain1D(id=1, x=0)
surf_right_C = F.SurfaceSubdomain1D(id=2, x=L)
model_C.subdomains = [vol_C, surf_left_C, surf_right_C]

# ── Species ───────────────────────────────────────────────────────────────────
tritium_C    = F.Species('T')
trapped_T1   = F.Species('T_trap1', mobile=False)   # intrinsic
trapped_T2   = F.Species('T_trap2', mobile=False)   # irradiation-induced

# Each ImplicitSpecies tracks its OWN empty sites
n_trap1 = 50.0    # intrinsic trap density  [mol/m³]
n_trap2 = 200.0   # radiation-induced trap density [mol/m³]  (more dense)
empty1  = F.ImplicitSpecies(n=n_trap1, others=[trapped_T1])
empty2  = F.ImplicitSpecies(n=n_trap2, others=[trapped_T2])

model_C.species = [tritium_C, trapped_T1, trapped_T2]

# ── Reactions ─────────────────────────────────────────────────────────────────
# Trap 1 — lower binding energy (intrinsic, easier to release)
reaction_1 = F.Reaction(
    reactant=[tritium_C, empty1], product=[trapped_T1],
    k_0=k_0_trap, E_k=E_k_trap,
    p_0=p_0_trap, E_p=1.50,    # E_p slightly lower → easier detrapping
    volume=vol_C,
)
# Trap 2 — higher binding energy (irradiation-induced, hard to release)
reaction_2 = F.Reaction(
    reactant=[tritium_C, empty2], product=[trapped_T2],
    k_0=k_0_trap, E_k=E_k_trap,
    p_0=p_0_trap, E_p=1.80,    # E_p higher → harder detrapping
    volume=vol_C,
)
model_C.reactions = [reaction_1, reaction_2]

# ── Retardation factors ────────────────────────────────────────────────────────
R1 = 1 + (k_0_trap * np.exp(-E_k_trap/(kB*T_K))) / (p_0_trap * np.exp(-1.50/(kB*T_K))) * n_trap1
R2 = (k_0_trap * np.exp(-E_k_trap/(kB*T_K))) / (p_0_trap * np.exp(-1.80/(kB*T_K))) * n_trap2
R_total = 1 + (R1 - 1) + R2
print(f"R from trap 1 alone  : {R1:.1f}")
print(f"R from trap 2 alone  : {1+R2:.1f}")
print(f"Total retardation R  : {R_total:.1f}")

model_C.temperature = T_K
model_C.boundary_conditions = [
    F.SievertsBC(
        subdomain=surf_left_C, S_0=S_0, E_S=E_S,
        pressure=P_up, species=tritium_C,
    ),
    F.DirichletBC(subdomain=surf_right_C, value=0.0, species=tritium_C),
]

# ── Exports ───────────────────────────────────────────────────────────────────
flux_C       = F.SurfaceFlux(surface=surf_right_C, field=tritium_C)
inv_C_mob    = F.TotalVolume(field=tritium_C,  volume=vol_C)
inv_C_trap1  = F.TotalVolume(field=trapped_T1, volume=vol_C)
inv_C_trap2  = F.TotalVolume(field=trapped_T2, volume=vol_C)

model_C.exports = [flux_C, inv_C_mob, inv_C_trap1, inv_C_trap2]

# ── Settings ──────────────────────────────────────────────────────────────────
tau_eff_C     = R_total * tau_diff
final_time_C  = 2 * tau_eff_C
model_C.settings = F.Settings(
    atol=1e10, rtol=1e-10,
    transient=True,
    final_time=final_time_C,
)
model_C.settings.stepsize = F.Stepsize(
    initial_value=tau_diff * 1e-3,
    growth_factor=1.3,
    cutback_factor=0.8,
    target_nb_iterations=4,
    max_stepsize=tau_eff_C / 20,
)

print(f"\nRunning Model C (2 traps, R={R_total:.0f}) to t = {final_time_C:.2e} s ...")
model_C.initialise()
model_C.run()
print("Model C done.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

t_C = np.array(inv_C_mob.t)
I_mob   = np.array(inv_C_mob.data)
I_trap1 = np.array(inv_C_trap1.data)
I_trap2 = np.array(inv_C_trap2.data)
I_total = I_mob + I_trap1 + I_trap2

ax.stackplot(
    t_C / tau_diff,
    I_mob, I_trap1, I_trap2,
    labels=['Mobile', 'Trap 1 (intrinsic)', 'Trap 2 (irradiation)'],
    colors=['steelblue', 'gold', 'tomato'],
    alpha=0.7,
)

ax.set(xlabel='t / τ_diff', ylabel='Inventory  [mol/m²]',
       title=f'Two-trap model — inventory breakdown (R_total = {R_total:.0f})')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('C_two_traps_inventory.png', dpi=120)
plt.show()

print(f"At steady state:")
print(f"  Mobile inventory  : {I_mob[-1]:.3e} mol/m²  ({100*I_mob[-1]/I_total[-1]:.1f}%)")
print(f"  Trap 1 inventory  : {I_trap1[-1]:.3e} mol/m²  ({100*I_trap1[-1]/I_total[-1]:.1f}%)")
print(f"  Trap 2 inventory  : {I_trap2[-1]:.3e} mol/m²  ({100*I_trap2[-1]/I_total[-1]:.1f}%)")
print(f"  TOTAL             : {I_total[-1]:.3e} mol/m²")

---
## 7. Summary and what's next

### What you've learned in this notebook

- The **retardation factor** $R = 1 + K_{eq}\, n_t$ tells you how much trapping slows the system
- Trapping does **not** change the steady-state permeation flux — it only increases inventory and delays breakthrough
- `F.ImplicitSpecies` is the correct way to represent empty trap sites (avoids solving an extra PDE)
- `F.Reaction` is the most flexible path; `F.Trap` is a convenient shorthand for simple cases
- Multiple traps simply add more `F.Reaction` + `F.ImplicitSpecies` pairs

### Key parameters for Li₂O trap calibration

The illustrative values used here should eventually be replaced with literature-calibrated data.  
Key references for Li₂O trapping: Tanifuji (1987), Dienes & Becker (1992), Moritani et al.

| Parameter | This notebook | Calibrated range (literature) |
|-----------|--------------|-------------------------------|
| $E_p$ (intrinsic) | 1.65 eV | 1.0–1.8 eV |
| $n_t$ (intrinsic) | 50–100 mol/m³ | 0.01–1% of Li₂O density |
| $p_0$ | 10¹³ s⁻¹ | 10¹² – 10¹³ s⁻¹ |

---

### Next notebook: **03 — Multi-material**

In reality, a breeding blanket has several layers (e.g. a tungsten first wall bonded to a Li₂O pebble bed).  
In Notebook 03 you will learn:

- How to define multiple `VolumeSubdomain1D` regions with different materials
- How FESTIM handles the **interface condition** (continuity of chemical potential → Sievert jump in concentration)
- How to extract quantities per-layer (partial inventories)
- Why a W capping layer can dramatically reduce tritium inventory

Then in **04** we add a temperature gradient, and in **05** we build the 3D pebble.